## Emotion Detection Function

In [1]:
from tensorflow.keras.models import load_model
from faster_whisper import WhisperModel

emotion_model = load_model("emotion_model.h5")

# Faster Whisper (optimized)
speech_model = WhisperModel(
    "base", 
    compute_type="int8", 
    cpu_threads=4
)

print("✅ Models Loaded")

config.json: 0.00B [00:00, ?B/s]

c:\Users\chinm\resumegen\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\chinm\.cache\huggingface\hub\models--Systran--faster-whisper-base. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


vocabulary.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.bin:   0%|          | 0.00/145M [00:00<?, ?B/s]

✅ Models Loaded


In [2]:
import cv2
import numpy as np
import threading

emotion_labels = ['Angry', 'Disgust', 'Fear', 'Happy', 'Sad', 'Surprise', 'Neutral']

face_cascade = cv2.CascadeClassifier(
    cv2.data.haarcascades + 'haarcascade_frontalface_default.xml'
)

print("✅ CV setup ready")

✅ CV setup ready


## emotion detection

In [3]:
def detect_emotion(frame):
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    faces = face_cascade.detectMultiScale(gray, 1.3, 5)

    emotions = []

    for (x, y, w, h) in faces:
        roi = gray[y:y+h, x:x+w]
        roi = cv2.resize(roi, (48, 48)) / 255.0
        roi = roi.reshape(1, 48, 48, 1)

        prediction = emotion_model.predict(roi, verbose=0)
        emotions.append(emotion_labels[np.argmax(prediction)])

    return emotions

## Video Emotion Analysis

In [4]:
import os

def analyze_video_emotion(video_path, output_dir):
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)

    cap = cv2.VideoCapture(video_path)
    emotion_list = []

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        emotion_list.extend(detect_emotion(frame))

    cap.release()
    return emotion_list

## Extract Audio

In [5]:
import subprocess

def extract_audio_from_video(video_path, audio_path="extracted_audio.wav"):
    command = [
        "ffmpeg", "-i", video_path,
        "-q:a", "0", "-map", "a", audio_path, "-y"
    ]

    result = subprocess.run(command, capture_output=True, text=True)

    if result.returncode != 0:
        print(" FFmpeg Error:", result.stderr)
        return None

    return audio_path

##  TRANSCRIPTION (Uses global speech_model)

In [6]:
def transcribe_audio(audio_path):
    print("Transcribing audio with Whisper...")

    segments, _ = speech_model.transcribe(audio_path)

    text = ""
    for segment in segments:
        text += segment.text

    return text

##  AUDIO FEATURES (Imports for Librosa)

In [7]:
import librosa

def analyze_audio_features(audio_path):
    y, sr = librosa.load(audio_path)

    volume = np.mean(np.abs(y))
    zcr = np.mean(librosa.feature.zero_crossing_rate(y))

    return {
        "confidence_score": float(min(volume * 10, 1.0)),
        "clarity_score": float(1 - zcr)
    }


## TEXT ANALYSIS (Import for TextBlob)

In [8]:
from textblob import TextBlob

def analyze_text_sentiment(text):
    blob = TextBlob(text)

    return {
        "sentiment": blob.sentiment.polarity,
        "confidence": 1 - blob.sentiment.subjectivity
    }

## Webcam Function(whisper real-time)

In [9]:
import sounddevice as sd
import queue
import threading
import time

SAMPLE_RATE = 16000
CHUNK_DURATION = 1       # 🔥 faster
OVERLAP = 0.3

audio_queue = queue.Queue()
current_transcript = "Listening..."

def audio_callback(indata, frames, time_info, status):
    audio_queue.put(indata.copy())


def audio_worker():
    global current_transcript
    import numpy as np

    buffer = np.zeros((0, 1), dtype=np.float32)

    while True:
        try:
            data = audio_queue.get()
            buffer = np.concatenate((buffer, data), axis=0)

            if len(buffer) >= SAMPLE_RATE * CHUNK_DURATION:

                chunk = buffer[:SAMPLE_RATE * CHUNK_DURATION]
                buffer = buffer[int(SAMPLE_RATE * (CHUNK_DURATION - OVERLAP)):]

                chunk = chunk.flatten()

                if np.max(np.abs(chunk)) > 0:
                    chunk = chunk / np.max(np.abs(chunk))

                segments, _ = speech_model.transcribe(chunk)

                text = ""
                for seg in segments:
                    text += seg.text

                if text.strip():
                    current_transcript = text.strip()

                time.sleep(0.1)  # prevents overload

        except Exception as e:
            print("Audio error:", e)


def start_audio():
    stream = sd.InputStream(
        samplerate=SAMPLE_RATE,
        channels=1,
        dtype='float32',
        callback=audio_callback
    )
    stream.start()

    threading.Thread(target=audio_worker, daemon=True).start()

    return stream


def real_time_webcam_emotion():

    stream = start_audio()
    cap = cv2.VideoCapture(0)

    print("🎥 Webcam started (Press 'q' to quit)")

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        emotions = detect_emotion(frame)

        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        faces = face_cascade.detectMultiScale(gray, 1.3, 5)

        for (x, y, w, h), emotion in zip(faces, emotions):
            cv2.rectangle(frame, (x, y), (x+w, y+h), (0,255,0), 2)
            cv2.putText(frame, emotion, (x, y-10),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0,255,0), 2)

        cv2.rectangle(frame, (0, frame.shape[0]-50),
                      (frame.shape[1], frame.shape[0]), (0,0,0), -1)

        cv2.putText(frame, f"You: {current_transcript}",
                    (10, frame.shape[0]-15),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7,
                    (255,255,255), 2)

        cv2.imshow("Live Emotion + FAST Transcript", frame)

        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

    cap.release()
    stream.stop()
    stream.close()
    cv2.destroyAllWindows()

## MAIN EXECUTION BLOCK

In [10]:
if __name__ == "__main__":

    print("="*50)
    print(" EMOTION ANALYSIS SYSTEM ")
    print("="*50)

    choice = input("Type 'video' or 'webcam': ").strip().lower()

    if choice == "webcam":
        real_time_webcam_emotion()

    elif choice == "video":

        video_path = r"C:\Users\chinm\Pictures\Camera Roll\video.mp4"
        output_dir = "emotion_analysis_output"
        audio_path = "extracted_audio.wav"

        emotions = analyze_video_emotion(video_path, output_dir)

        if extract_audio_from_video(video_path, audio_path):
            text = transcribe_audio(audio_path)

            audio_result = analyze_audio_features(audio_path)
            nlp_result = analyze_text_sentiment(text)

            dominant_emotion = max(set(emotions), key=emotions.count)

            print("\n📊 FINAL REPORT")
            print("="*40)
            print("Dominant Emotion:", dominant_emotion)
            print("Transcript:", text)
            print("Audio:", audio_result)
            print("Text:", nlp_result)

    else:
        print("Invalid input")

 EMOTION ANALYSIS SYSTEM 
🎥 Webcam started (Press 'q' to quit)
